# **Easy/Medium DIfficulaty features**

In this nootebook, I created 9 easy/medium difficulty features. They were tested by training a Random Forest Classifier. It achieved an accuracy of 64.5% when tested on 172 games and trained only on data of the past 2.5 years (older data is still used to calculate features for the train and val games). However, when using other seeds for RFC, accuracy can decrease of a few percents (e. g. 62.5%).

**Imports**

In [365]:
!pip install xgboost

import pandas as pd
import numpy as np
from datetime import date
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

**Importing data and preparing it for models.**

In [366]:
df = pd.read_csv('Data_Files/epl-training.csv')
df['Date'] = pd.to_datetime(df['Date'], format="%d/%m/%Y")
df = df.sort_values("Date").reset_index(drop=True)

df_train = df.copy()

In [367]:
#Convert date to number of days until 31/01/2026 (date when the games are going to be played)
target = pd.to_datetime("31/01/2026", format="%d/%m/%Y")
df_train['NumDays'] = (target - df_train['Date']).dt.days

In [368]:
#Creates a new data frame with only the relevant data which will be directly used
#for training (the features will be calculated from the old df,
#new df is only for training and val)
df_new_features = df_train[['Date', 'NumDays', 'HomeTeam', 'AwayTeam', 'FTR']].copy()

**First feature: H2H**

This corresponds to a parameter representing how well the home team performed against the away team in the past. There is a coefficient associated to the date of each game between the two teams (high for recent games and low for older). This coefficient is decaying exponentially with the time passed since the game and its parameter k is selected by testing various values and picking the best performing one.

In [369]:
#Finds all the past games between the two teams
def get_past_matches(df, home, away, date):
    mask = (((df['HomeTeam'] == home) & (df['AwayTeam'] == away)) | ((df['HomeTeam'] == away) & (df['AwayTeam'] == home))) & (df['NumDays'] > date)
    return df[mask]

#computes h2h for a specific game
def compute_h2h(df, home, away, date, k=0.001, j=0.8, l=0.1):
  matches = get_past_matches(df, home, away, date)
  if matches.empty: #default value in case no games happened before
      return 0.5

  weighted_sum = 0
  count=0

  #Goes through each games which happend before this one to compute its past h2h
  for _, row in matches.iterrows():

      age_days = row['NumDays'] - date
      w = np.exp(-k * age_days)

      # 1 point for a win, 0.5 for draw and 0 for loss
      #j is a factor giving higher weight for games played at home for the home team and more weight for away games for the away team
      if row['HomeTeam'] == home:
          if row['FTR'] == 'H':   res = 1.0
          elif row['FTR'] == 'D': res = 0.5
          else: res = 0.0
          weighted_sum += res * w
          count+=w
      #l gives a better score for the team if the team was the away side in a specific past game
      else:
          if row['FTR'] == 'A':   res = 1.0
          elif row['FTR'] == 'D': res = 0.5 + l
          else: res = 0.0
          weighted_sum += res * w * j
          count+=w*j

  return weighted_sum/count

#Initialise the column
df_new_features["H2H"] = 0.0


**Feature 2**

Global recent form: computes a score given by the number of W/D/L in recent games for each of the two team

**Feature 3**

Same thing but with goal difference: weighted average


Here the weight is calculated using a logistic function 1/(1+exp(-kt)), which is more suitable than exponential.




In [370]:
#This function returns the K last games of a team before a specific fixture
def get_last_matches(df, team, date, K):
  mask = (((df['HomeTeam'] == team) | (df['AwayTeam'] == team)) &
          (df['NumDays'] > date))

  past = df[mask].head(K)
  return past

In [371]:
#This computes the weight given to a game when calculating those features
#k and m are parameters tuned arbitrarily
#zmax is used to avoid overflow for games that happened too long ago
def logistic_weight(age_days, k, m, zmax=500):
    z = k * (age_days - m)
    z = np.clip(z, -zmax, zmax)
    return 1 / (1 + np.exp(z))

#Computes recent form, similar way to feature 1
#Takes the 10 last games and apply weights
def compute_recent_form(df, team, date, K=10, k=0.15, m=30, j=0.8, l=0.1):
    past = get_last_matches(df, team, date, K)
    #default values
    if past.empty:
        return 0.5, 0

    weighted_sum = 0
    weight_total = 0

    gd_weighted_sum = 0
    gd_weight_total = 0

    for _, row in past.iterrows():
        age_days = row['NumDays'] - date
        w=logistic_weight(age_days, k, m)

        if row['HomeTeam'] == team:
          if row['FTR'] == 'H':   res = 1.0
          elif row['FTR'] == 'D': res = 0.5
          else: res = 0.0
          weighted_sum += res * w
          weight_total += w

          gd = row['FTHG'] - row['FTAG']
          gd_weighted_sum += gd * w
          gd_weight_total += w

        else:
          if row['FTR'] == 'A':   res = 1.0 + l
          elif row['FTR'] == 'D': res = 0.5 + l
          else: res = l
          weighted_sum += res * w * j
          weight_total += w*j

          gd = row['FTAG'] - row['FTHG']
          gd_weighted_sum += gd * w*j
          gd_weight_total += w*j

    if weight_total == 0:
        return 0.5, 0.0

    recent_form = weighted_sum / weight_total
    recent_gd = gd_weighted_sum / gd_weight_total if gd_weight_total > 0 else 0.0

    return recent_form, recent_gd




**Feature 4**

Home win rate in the past year

**Feature 5**

Away win rate in the past year

In [372]:
def compute_home_win_rate(df, team, date):
    mask = (
        (df["HomeTeam"] == team)
        & (df["NumDays"] > date)
        & ((df["NumDays"] - date) < 365)
    )
    past = df[mask]
    if past.empty:
        return 0.5
    wins = (past["FTR"] == 'H').sum()
    return wins / len(past)


def compute_away_win_rate(df, team, date):
    mask = (
        (df["AwayTeam"] == team)
        & (df["NumDays"] > date)
        & ((df["NumDays"] - date) < 365)
    )
    past = df[mask]
    if past.empty:
        return 0.5
    wins = (past["FTR"] == 'A').sum()
    return wins / len(past)

**Features 67 8 and 9**

Simple statistics:

- gf: Goals scored by the team in the past 10 games

- ga: Goals against by the team in the past 10 games

- sf: Shots scored by the team in the past 10 games

- sa: Shots against by the team in the past 10 games

In [373]:
def compute_recent_simple_stats(df, team, date, K=10):
    past = get_last_matches(df, team, date, K)
    if past.empty:
        return 0.0, 0.0, 0.0, 0.0

    home = past[past["HomeTeam"] == team]
    away = past[past["AwayTeam"] == team]

    gf = home["FTHG"].sum() + away["FTAG"].sum()
    ga = home["FTAG"].sum() + away["FTHG"].sum()
    sf = home["HS"].sum()   + away["AS"].sum()
    sa = home["AS"].sum()   + away["HS"].sum()

    n = len(past)
    return gf / n, ga / n, sf / n, sa / n

**Computing the new features**

In [374]:
#Initialising the new colums of the df used for training the model

df_new_features["RecentWH"] = 0.0
df_new_features["RecentWA"] = 0.0
df_new_features["RecentGDH"] = 0.0
df_new_features["RecentGDA"] = 0.0
df_new_features["HomeWinRate"] = 0.0
df_new_features["AwayWinRate"] = 0.0

df_new_features["AvgGF_H"] = 0.0
df_new_features["AvgGA_H"] = 0.0
df_new_features["AvgSF_H"] = 0.0
df_new_features["AvgSA_H"] = 0.0

df_new_features["AvgGF_A"] = 0.0
df_new_features["AvgGA_A"] = 0.0
df_new_features["AvgSF_A"] = 0.0
df_new_features["AvgSA_A"] = 0.0

In [375]:
# Caluclating the features for each row
# This code is really slow, can take up to 2 minutes to run
# Could be massively optimised
for i, game in df_train.iterrows():
    df_new_features.loc[i, "H2H"] = compute_h2h(df_train, game["HomeTeam"], game["AwayTeam"], game["NumDays"])
    df_new_features.loc[i, "RecentWH"], df_new_features.loc[i, "RecentGDH"] = compute_recent_form(df_train, game["HomeTeam"], game["NumDays"])
    df_new_features.loc[i, "RecentWA"], df_new_features.loc[i, "RecentGDA"] = compute_recent_form(df_train, game["AwayTeam"], game["NumDays"])
    df_new_features.loc[i, "HomeWinRate"] = compute_home_win_rate(df_train, game["HomeTeam"], game["NumDays"])
    df_new_features.loc[i, "AwayWinRate"] = compute_away_win_rate(df_train, game["AwayTeam"], game["NumDays"])
    df_new_features.loc[i, ["AvgGF_A", "AvgGA_A", "AvgSF_A", "AvgSA_A"]] = compute_recent_simple_stats(df_train, game["AwayTeam"], game["NumDays"])
    df_new_features.loc[i, ["AvgGF_H", "AvgGA_H", "AvgSF_H", "AvgSA_H"]] = compute_recent_simple_stats(df_train, game["HomeTeam"], game["NumDays"])

In [376]:
# This gives more features to the model, such as form difference
# They can appear redundant since it's only doing simple arithmetics on existing features but is useful for models such as binary trees
df_new_features["FormDiff"]         = df_new_features["RecentWH"]  - df_new_features["RecentWA"]
df_new_features["GDDiff"]           = df_new_features["RecentGDH"] - df_new_features["RecentGDA"]
df_new_features["ShotDiff"]         = df_new_features["AvgSF_H"]   - df_new_features["AvgSF_A"]
df_new_features["ShotAllowedDiff"]  = df_new_features["AvgSA_H"]   - df_new_features["AvgSA_A"]
df_new_features["AttackBalance"]    = (df_new_features["AvgGF_H"] + df_new_features["AvgGF_A"]) - (df_new_features["AvgGA_H"] + df_new_features["AvgGA_A"])
df_new_features["DefenseBalance"]   = (df_new_features["AvgGA_H"] + df_new_features["AvgGA_A"])

In [377]:
df_new_features

,Date,NumDays,HomeTeam,AwayTeam,FTR,H2H,RecentWH,RecentWA,RecentGDH,RecentGDA,...,AvgGF_A,AvgGA_A,AvgSF_A,AvgSA_A,FormDiff,GDDiff,ShotDiff,ShotAllowedDiff,AttackBalance,DefenseBalance
0,2000-08-19,9296,Charlton,Man City,H,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
1,2000-08-19,9296,Chelsea,West Ham,H,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
2,2000-08-19,9296,Coventry,Middlesbrough,A,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
3,2000-08-19,9296,Derby,Southampton,D,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
4,2000-08-19,9296,Leeds,Everton,H,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9595,2025-05-25,251,Ipswich,West Ham,A,0.000282,0.588889,0.388889,0.266667,-0.222222,...,1.2,1.4,13.2,11.0,0.200000,0.488889,-0.7,-0.7,0.1,2.5
9596,2025-05-25,251,Fulham,Man City,A,0.004437,0.502174,0.544444,-0.152174,0.044444,...,1.4,1.4,10.5,12.1,-0.042271,-0.196618,-0.5,-2.7,-0.2,2.6
9597,2025-05-25,251,Bournemouth,Leicester,H,0.562270,0.355556,0.644444,-0.955556,0.066667,...,0.7,0.6,9.4,12.2,-0.288889,-1.022222,1.1,-1.4,-0.9,2.8
9598,2025-05-25,251,Liverpool,Crystal Palace,D,0.729806,0.722222,0.355556,0.466667,-0.355556,...,1.1,1.5,10.2,14.8,0.366667,0.822222,2.3,-4.8,0.0,2.8


# **Hard DIfficulaty features**

### Team Form and Rolling Averages (Last k matches)

In [378]:
def create_team_history(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a unified team history DataFrame from match data.
    
    Combines home and away records into a single timeline per team,
    with goals, shots, and other stats from that team's perspective.
    
    Parameters
    ----------
    df : pd.DataFrame
        Match data with columns: Date, HomeTeam, AwayTeam, FTHG, FTAG, FTR,
        HS, AS, HST, AST, HF, AF, HC, AC
    
    Returns
    -------
    pd.DataFrame
        Team-level history sorted by (Team, Date) with columns:
        Date, Team, GF, GA, FTR, Shots, ShotsOnTarget, Fouls, Corners,
        is_home, GoalDiff, Points
    """
    # Home team perspective
    home = df[["Date", "HomeTeam", "FTHG", "FTAG", "FTR", "HS", "HST", "HF", "HC"]].copy()
    home.rename(columns={
        "HomeTeam": "Team",
        "FTHG": "GF",
        "FTAG": "GA",
        "HS": "Shots",
        "HST": "ShotsOnTarget",
        "HF": "Fouls",
        "HC": "Corners"
    }, inplace=True)
    home["is_home"] = 1

    # Away team perspective
    away = df[["Date", "AwayTeam", "FTHG", "FTAG", "FTR", "AS", "AST", "AF", "AC"]].copy()
    away.rename(columns={
        "AwayTeam": "Team",
        "FTAG": "GF",
        "FTHG": "GA",
        "AS": "Shots",
        "AST": "ShotsOnTarget",
        "AF": "Fouls",
        "AC": "Corners"
    }, inplace=True)
    away["is_home"] = 0

    # Combine and sort
    team_history = (
        pd.concat([home, away], ignore_index=True)
        .sort_values(["Team", "Date"])
        .reset_index(drop=True)
    )

    # Derived columns
    team_history["GoalDiff"] = team_history["GF"] - team_history["GA"]
    team_history["Points"] = (
        (team_history["GF"] > team_history["GA"]).astype(int) * 3
        + (team_history["GF"] == team_history["GA"]).astype(int) * 1
        # 3 points for win, 1 for draw, 0 for loss
        # losses automatically get 0
    )

    return team_history

In [379]:
team_history = create_team_history(df_train)
team_history

,Date,Team,GF,GA,FTR,Shots,ShotsOnTarget,Fouls,Corners,is_home,GoalDiff,Points
0,2000-08-19,Arsenal,0,1,H,14,7,21,9,0,-1,0
1,2000-08-21,Arsenal,2,0,H,17,12,25,10,1,2,3
2,2000-08-26,Arsenal,5,3,H,18,9,12,8,1,2,3
3,2000-09-06,Arsenal,2,2,D,13,5,22,6,0,0,1
4,2000-09-09,Arsenal,1,1,D,18,11,13,10,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
19195,2025-04-26,Wolves,3,0,H,20,6,4,7,1,3,3
19196,2025-05-02,Wolves,0,1,H,6,1,5,11,0,-1,0
19197,2025-05-10,Wolves,0,2,A,10,3,7,8,1,-2,0
19198,2025-05-20,Wolves,2,4,H,12,3,10,8,0,-2,0


In [380]:
k = 5  # number of previous matches to look back over

# Group by team, because we want each team’s history separately
grp = team_history.groupby("Team", group_keys=False)

In [381]:
# Rolling sums for GF, GA, Points over last k matches (excluding current match)
for col in ["GF", "GA", "Points"]:
    team_history[f"{col}_sum_last_{k}"] = grp[col].apply(
        lambda s: s.shift(1).rolling(window=k, min_periods=1).sum()
    )

team_history

,Date,Team,GF,GA,FTR,Shots,ShotsOnTarget,Fouls,Corners,is_home,GoalDiff,Points,GF_sum_last_5,GA_sum_last_5,Points_sum_last_5
0,2000-08-19,Arsenal,0,1,H,14,7,21,9,0,-1,0,NaN,NaN,NaN
1,2000-08-21,Arsenal,2,0,H,17,12,25,10,1,2,3,0.0,1.0,0.0
2,2000-08-26,Arsenal,5,3,H,18,9,12,8,1,2,3,2.0,1.0,3.0
3,2000-09-06,Arsenal,2,2,D,13,5,22,6,0,0,1,7.0,4.0,6.0
4,2000-09-09,Arsenal,1,1,D,18,11,13,10,0,0,1,9.0,6.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19195,2025-04-26,Wolves,3,0,H,20,6,4,7,1,3,3,10.0,4.0,15.0
19196,2025-05-02,Wolves,0,1,H,6,1,5,11,0,-1,0,11.0,3.0,15.0
19197,2025-05-10,Wolves,0,2,A,10,3,7,8,1,-2,0,10.0,4.0,12.0
19198,2025-05-20,Wolves,2,4,H,12,3,10,8,0,-2,0,8.0,5.0,9.0


In [382]:
# Rolling means for GF, GA, GoalDiff, Shots, ShotsOnTarget, Corners over last k matches (excluding current match)
for col in ["GF", "GA", "GoalDiff", "Shots", "ShotsOnTarget", "Corners"]:
    team_history[f"{col}_mean_last_{k}"] = grp[col].apply(
        lambda s: s.shift(1).rolling(window=k, min_periods=1).mean()
    )

team_history

,Date,Team,GF,GA,FTR,Shots,ShotsOnTarget,Fouls,Corners,is_home,...,Points,GF_sum_last_5,GA_sum_last_5,Points_sum_last_5,GF_mean_last_5,GA_mean_last_5,GoalDiff_mean_last_5,Shots_mean_last_5,ShotsOnTarget_mean_last_5,Corners_mean_last_5
0,2000-08-19,Arsenal,0,1,H,14,7,21,9,0,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-08-21,Arsenal,2,0,H,17,12,25,10,1,...,3,0.0,1.0,0.0,0.000000,1.000000,-1.00,14.000000,7.000000,9.00
2,2000-08-26,Arsenal,5,3,H,18,9,12,8,1,...,3,2.0,1.0,3.0,1.000000,0.500000,0.50,15.500000,9.500000,9.50
3,2000-09-06,Arsenal,2,2,D,13,5,22,6,0,...,1,7.0,4.0,6.0,2.333333,1.333333,1.00,16.333333,9.333333,9.00
4,2000-09-09,Arsenal,1,1,D,18,11,13,10,0,...,1,9.0,6.0,7.0,2.250000,1.500000,0.75,15.500000,8.250000,8.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19195,2025-04-26,Wolves,3,0,H,20,6,4,7,1,...,3,10.0,4.0,15.0,2.000000,0.800000,1.20,10.600000,3.800000,15.00
19196,2025-05-02,Wolves,0,1,H,6,1,5,11,0,...,0,11.0,3.0,15.0,2.200000,0.600000,1.60,13.600000,4.400000,13.80
19197,2025-05-10,Wolves,0,2,A,10,3,7,8,1,...,0,10.0,4.0,12.0,2.000000,0.800000,1.20,13.000000,4.200000,12.80
19198,2025-05-20,Wolves,2,4,H,12,3,10,8,0,...,0,8.0,5.0,9.0,1.600000,1.000000,0.60,10.600000,3.400000,11.20


At this point, team_history has, for each (Team, Date), features like:
    `GF_sum_last_5`, `GA_sum_last_5`, `Points_sum_last_5`,
    `GF_mean_last_5`, `GA_mean_last_5`, `GoalDiff_mean_last_5`, ...

The first few matches for each team will have NaNs
because they don't have k previous games.

In [383]:
# Pick just the columns we care about
rolling_cols = [c for c in team_history.columns if f"last_{k}" in c]
base_cols = ["Date", "Team", "is_home"]

team_feats = team_history[base_cols + rolling_cols].copy()
team_feats

,Date,Team,is_home,GF_sum_last_5,GA_sum_last_5,Points_sum_last_5,GF_mean_last_5,GA_mean_last_5,GoalDiff_mean_last_5,Shots_mean_last_5,ShotsOnTarget_mean_last_5,Corners_mean_last_5
0,2000-08-19,Arsenal,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-08-21,Arsenal,1,0.0,1.0,0.0,0.000000,1.000000,-1.00,14.000000,7.000000,9.00
2,2000-08-26,Arsenal,1,2.0,1.0,3.0,1.000000,0.500000,0.50,15.500000,9.500000,9.50
3,2000-09-06,Arsenal,0,7.0,4.0,6.0,2.333333,1.333333,1.00,16.333333,9.333333,9.00
4,2000-09-09,Arsenal,0,9.0,6.0,7.0,2.250000,1.500000,0.75,15.500000,8.250000,8.25
...,...,...,...,...,...,...,...,...,...,...,...,...
19195,2025-04-26,Wolves,1,10.0,4.0,15.0,2.000000,0.800000,1.20,10.600000,3.800000,15.00
19196,2025-05-02,Wolves,0,11.0,3.0,15.0,2.200000,0.600000,1.60,13.600000,4.400000,13.80
19197,2025-05-10,Wolves,1,10.0,4.0,12.0,2.000000,0.800000,1.20,13.000000,4.200000,12.80
19198,2025-05-20,Wolves,0,8.0,5.0,9.0,1.600000,1.000000,0.60,10.600000,3.400000,11.20


In [384]:
# Split into home and away versions
home_feats = (
    team_feats[team_feats["is_home"] == 1]
    .drop(columns=["is_home"])
    .sort_values(["Team", "Date"])
)

away_feats = (
    team_feats[team_feats["is_home"] == 0]
    .drop(columns=["is_home"])
    .sort_values(["Team", "Date"])
)

In [385]:
# Deduplicate by key: (Date, Team)
home_feats = home_feats.drop_duplicates(subset=["Date", "Team"], keep="last")
away_feats = away_feats.drop_duplicates(subset=["Date", "Team"], keep="last")

In [386]:
# Make a fresh copy of the original match-level df (with Date as datetime)
matches_with_form = df_train.copy()
#matches_with_form["Date"] = pd.to_datetime(matches_with_form["Date"], dayfirst=True)

In [387]:
# Merge home form: match (Date, HomeTeam) with (home_Date, home_Team)
matches_with_form = matches_with_form.merge(
    home_feats.add_prefix("home_"),
    left_on=["Date", "HomeTeam"],
    right_on=["home_Date", "home_Team"],
    how="left"
)

# Merge away form: match (Date, AwayTeam) with (away_Date, away_Team)
matches_with_form = matches_with_form.merge(
    away_feats.add_prefix("away_"),
    left_on=["Date", "AwayTeam"],
    right_on=["away_Date", "away_Team"],
    how="left"
)

In [388]:
# Drop duplicate join keys (we still have the original Date/HomeTeam/AwayTeam)
matches_with_form = matches_with_form.drop(
    columns=["home_Date", "home_Team", "away_Date", "away_Team"]
)

In [389]:
matches_with_form["points_form_diff_last_5"] = (
    matches_with_form["home_Points_sum_last_5"]
    - matches_with_form["away_Points_sum_last_5"]
)

matches_with_form["goal_diff_form_last_5"] = (
    matches_with_form["home_GF_sum_last_5"] - matches_with_form["home_GA_sum_last_5"]
    - (matches_with_form["away_GF_sum_last_5"] - matches_with_form["away_GA_sum_last_5"])
)

In [390]:
matches_with_form

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,away_GA_sum_last_5,away_Points_sum_last_5,away_GF_mean_last_5,away_GA_mean_last_5,away_GoalDiff_mean_last_5,away_Shots_mean_last_5,away_ShotsOnTarget_mean_last_5,away_Corners_mean_last_5,points_form_diff_last_5,goal_diff_form_last_5
0,2000-08-19,Charlton,Man City,4,0,H,2,0,H,Rob Harris,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-08-19,Chelsea,West Ham,4,2,H,1,0,H,Graham Barber,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-08-19,Coventry,Middlesbrough,1,3,A,1,1,D,Barry Knight,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-08-19,Derby,Southampton,2,2,D,1,2,A,Andy D'Urso,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-08-19,Leeds,Everton,2,0,H,2,0,H,Dermot Gallagher,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9595,2025-05-25,Ipswich,West Ham,1,3,A,0,1,A,T Robinson,...,7.0,5.0,1.4,1.4,0.0,12.0,3.8,13.6,-4.0,-10.0
9596,2025-05-25,Fulham,Man City,0,2,A,0,1,A,A Madley,...,2.0,13.0,1.6,0.4,1.2,14.6,5.0,7.4,-7.0,-8.0
9597,2025-05-25,Bournemouth,Leicester,2,0,H,0,0,D,L Smith,...,6.0,7.0,1.2,1.2,0.0,9.2,3.0,13.2,-2.0,-2.0
9598,2025-05-25,Liverpool,Crystal Palace,1,1,D,0,1,A,D England,...,5.0,9.0,1.8,1.0,0.8,15.4,5.4,10.4,-2.0,-2.0


### Exponentially Weighted Moving Averages 

In [391]:
team_history = create_team_history(df_train)
team_history

,Date,Team,GF,GA,FTR,Shots,ShotsOnTarget,Fouls,Corners,is_home,GoalDiff,Points
0,2000-08-19,Arsenal,0,1,H,14,7,21,9,0,-1,0
1,2000-08-21,Arsenal,2,0,H,17,12,25,10,1,2,3
2,2000-08-26,Arsenal,5,3,H,18,9,12,8,1,2,3
3,2000-09-06,Arsenal,2,2,D,13,5,22,6,0,0,1
4,2000-09-09,Arsenal,1,1,D,18,11,13,10,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
19195,2025-04-26,Wolves,3,0,H,20,6,4,7,1,3,3
19196,2025-05-02,Wolves,0,1,H,6,1,5,11,0,-1,0
19197,2025-05-10,Wolves,0,2,A,10,3,7,8,1,-2,0
19198,2025-05-20,Wolves,2,4,H,12,3,10,8,0,-2,0


In [392]:
# More recent matches are weighted more heavily.
# span controls decay speed; smaller span = faster decay.
span = 8  # hyperparameter; you can tune this

grp = team_history.groupby("Team", group_keys=False)

def ewm_shifted(col_name: str, span: int) -> pd.Series:
    """
    For each team, compute EWM over *past* matches only:
    - shift(1) so current match is NOT included (no leakage)
    - ewm(span=span) to weight recent history more.
    """
    return grp[col_name].apply(
        lambda s: s.shift(1).ewm(span=span, adjust=False).mean()
    )

# Choose which stats to smooth
ewm_source_cols = ["GF", "GA", "GoalDiff", "Points",
                   "Shots", "ShotsOnTarget", "Fouls", "Corners"]

for col in ewm_source_cols:
    team_history[f"{col}_ewm"] = ewm_shifted(col, span)

In [393]:
team_history

,Date,Team,GF,GA,FTR,Shots,ShotsOnTarget,Fouls,Corners,is_home,GoalDiff,Points,GF_ewm,GA_ewm,GoalDiff_ewm,Points_ewm,Shots_ewm,ShotsOnTarget_ewm,Fouls_ewm,Corners_ewm
0,2000-08-19,Arsenal,0,1,H,14,7,21,9,0,-1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-08-21,Arsenal,2,0,H,17,12,25,10,1,2,3,0.000000,1.000000,-1.000000,0.000000,14.000000,7.000000,21.000000,9.000000
2,2000-08-26,Arsenal,5,3,H,18,9,12,8,1,2,3,0.444444,0.777778,-0.333333,0.666667,14.666667,8.111111,21.888889,9.222222
3,2000-09-06,Arsenal,2,2,D,13,5,22,6,0,0,1,1.456790,1.271605,0.185185,1.185185,15.407407,8.308642,19.691358,8.950617
4,2000-09-09,Arsenal,1,1,D,18,11,13,10,0,0,1,1.577503,1.433471,0.144033,1.144033,14.872428,7.573388,20.204390,8.294925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19195,2025-04-26,Wolves,3,0,H,20,6,4,7,1,3,3,1.745443,0.947021,0.798421,2.435036,11.033330,3.938660,3.282997,14.389498
19196,2025-05-02,Wolves,0,1,H,6,1,5,11,0,-1,0,2.024233,0.736572,1.287661,2.560584,13.025923,4.396735,3.442331,12.747387
19197,2025-05-10,Wolves,0,2,A,10,3,7,8,1,-2,0,1.574404,0.795112,0.779292,1.991565,11.464607,3.641905,3.788479,12.359079
19198,2025-05-20,Wolves,2,4,H,12,3,10,8,0,-2,0,1.224536,1.062865,0.161671,1.548995,11.139139,3.499260,4.502151,11.390395


In [394]:
# Columns we need from team_history to join back
ewm_cols = [c for c in team_history.columns if c.endswith("_ewm")]
base_cols = ["Date", "Team", "is_home"]

team_feats = team_history[base_cols + ewm_cols].copy()

# Split into home and away feature tables
home_feats = team_feats[team_feats["is_home"] == 1].drop(columns=["is_home"])
away_feats = team_feats[team_feats["is_home"] == 0].drop(columns=["is_home"])

# Prefix columns so they stay distinct
home_feats = home_feats.add_prefix("home_")
away_feats = away_feats.add_prefix("away_")

# Fresh copy of the original match-level data
matches_with_ewm = df_train.copy()
#matches_with_ewm["Date"] = pd.to_datetime(matches_with_ewm["Date"], dayfirst=True)

# Merge home EWMA features: (Date, HomeTeam) <-> (home_Date, home_Team)
matches_with_ewm = matches_with_ewm.merge(
    home_feats,
    left_on=["Date", "HomeTeam"],
    right_on=["home_Date", "home_Team"],
    how="left"
)

# Merge away EWMA features: (Date, AwayTeam) <-> (away_Date, away_Team)
matches_with_ewm = matches_with_ewm.merge(
    away_feats,
    left_on=["Date", "AwayTeam"],
    right_on=["away_Date", "away_Team"],
    how="left"
)

# Remove duplicate key columns from the merge
matches_with_ewm = matches_with_ewm.drop(
    columns=["home_Date", "home_Team", "away_Date", "away_Team"]
)

matches_with_ewm["ewm_points_diff"] = (
    matches_with_ewm["home_Points_ewm"] - matches_with_ewm["away_Points_ewm"]
)

matches_with_ewm["ewm_goal_diff_diff"] = (
    matches_with_ewm["home_GoalDiff_ewm"] - matches_with_ewm["away_GoalDiff_ewm"]
)


In [395]:
matches_with_ewm.columns

Index(['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG',
       'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HF', 'AF',
       'HY', 'AY', 'HR', 'AR', 'NumDays', 'home_GF_ewm', 'home_GA_ewm',
       'home_GoalDiff_ewm', 'home_Points_ewm', 'home_Shots_ewm',
       'home_ShotsOnTarget_ewm', 'home_Fouls_ewm', 'home_Corners_ewm',
       'away_GF_ewm', 'away_GA_ewm', 'away_GoalDiff_ewm', 'away_Points_ewm',
       'away_Shots_ewm', 'away_ShotsOnTarget_ewm', 'away_Fouls_ewm',
       'away_Corners_ewm', 'ewm_points_diff', 'ewm_goal_diff_diff'],
      dtype='object')

### ELO Ratings

In [396]:
def add_elo_features(
    df: pd.DataFrame,
    base_rating: float = 1500.0,
    k_factor: float = 20.0,
    home_advantage: float = 50.0
):
    """
    Compute ELO ratings over time and attach, for each match:
      - home_elo_pre: rating of HomeTeam *before* this match
      - away_elo_pre: rating of AwayTeam *before* this match
      - elo_diff_pre: home_elo_pre - away_elo_pre

    Parameters
    ----------
    df : DataFrame with at least ['Date','HomeTeam','AwayTeam','FTR']
    base_rating : starting ELO for all teams
    k_factor : learning rate for rating updates
    home_advantage : rating points added to home team when computing expectation

    Returns
    -------
    df_with_elo : DataFrame copy with new ELO columns
    final_ratings : dict {team_name: final_elo_after_last_match}
    """

    df = df.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    # Current ratings for each team
    ratings = {}

    # We’ll store pre-match ratings here
    home_elo_pre = []
    away_elo_pre = []

    def result_to_scores(ftr: str):
        """Map FTR (H/D/A) to ELO scores (S_home, S_away)."""
        if ftr == 'H':
            return 1.0, 0.0
        elif ftr == 'A':
            return 0.0, 1.0
        else:  # Draw
            return 0.5, 0.5

    # Iterate through matches in time order
    for _, row in df.iterrows():
        home = row["HomeTeam"]
        away = row["AwayTeam"]
        ftr = row["FTR"]

        # Get current ratings (default to base_rating the first time we see a team)
        R_h = ratings.get(home, base_rating)
        R_a = ratings.get(away, base_rating)

        # Save PRE-match ratings as features
        home_elo_pre.append(R_h)
        away_elo_pre.append(R_a)

        # --------- ELO UPDATE STEP ---------
        # Expected home score (with home advantage)
        exp_home = 1.0 / (1.0 + 10 ** (-(R_h + home_advantage - R_a) / 400.0))
        exp_away = 1.0 - exp_home

        # Actual scores from result
        S_h, S_a = result_to_scores(ftr)

        # New ratings
        R_h_new = R_h + k_factor * (S_h - exp_home)
        R_a_new = R_a + k_factor * (S_a - exp_away)

        # Store updated ratings
        ratings[home] = R_h_new
        ratings[away] = R_a_new

    # Attach pre-match ratings as columns
    df["home_elo_pre"] = home_elo_pre
    df["away_elo_pre"] = away_elo_pre
    df["elo_diff_pre"] = df["home_elo_pre"] - df["away_elo_pre"]

    return df, ratings


# Run on your training data
train_with_elo, final_elo_ratings = add_elo_features(df_train)

df_new_features["home_elo_pre"] = train_with_elo["home_elo_pre"]
df_new_features["away_elo_pre"] = train_with_elo["away_elo_pre"]
df_new_features["elo_diff_pre"] = train_with_elo["elo_diff_pre"]

In [397]:
df_new_features

,Date,NumDays,HomeTeam,AwayTeam,FTR,H2H,RecentWH,RecentWA,RecentGDH,RecentGDA,...,AvgSA_A,FormDiff,GDDiff,ShotDiff,ShotAllowedDiff,AttackBalance,DefenseBalance,home_elo_pre,away_elo_pre,elo_diff_pre
0,2000-08-19,9296,Charlton,Man City,H,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,1500.000000,1500.000000,0.000000
1,2000-08-19,9296,Chelsea,West Ham,H,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,1500.000000,1500.000000,0.000000
2,2000-08-19,9296,Coventry,Middlesbrough,A,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,1500.000000,1500.000000,0.000000
3,2000-08-19,9296,Derby,Southampton,D,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,1500.000000,1500.000000,0.000000
4,2000-08-19,9296,Leeds,Everton,H,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,1500.000000,1500.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9595,2025-05-25,251,Ipswich,West Ham,A,0.000282,0.588889,0.388889,0.266667,-0.222222,...,11.0,0.200000,0.488889,-0.7,-0.7,0.1,2.5,1351.780976,1772.479113,-420.698137
9596,2025-05-25,251,Fulham,Man City,A,0.004437,0.502174,0.544444,-0.152174,0.044444,...,12.1,-0.042271,-0.196618,-0.5,-2.7,-0.2,2.6,1677.011651,1577.256487,99.755164
9597,2025-05-25,251,Bournemouth,Leicester,H,0.562270,0.355556,0.644444,-0.955556,0.066667,...,12.2,-0.288889,-1.022222,1.1,-1.4,-0.9,2.8,1607.715011,1673.309574,-65.594563
9598,2025-05-25,251,Liverpool,Crystal Palace,D,0.729806,0.722222,0.355556,0.466667,-0.355556,...,14.8,0.366667,0.822222,2.3,-4.8,0.0,2.8,1549.451178,1695.466458,-146.015280


# **Hardest DIfficulaty features**

In [398]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


### Creating a (Per Match) Goal Difference Feature
The Goal Difference feature will be defined as: 

<center><i>GD = FTHG - FTAG</i></center>

This will quantify how many more or less goals the home team was able to get by the away team. It is important to emphasize that this is a per-match feature and not to be confused with the conventional goal-difference aggregate feature in football league tables.

**Why is this feature useful?**

It will be used in later modelling features such as:
- Quickly interpreting a match's 'intensity'.
- Estimating a team's attack and defense strength in the Poisson GLM.
- Calculating performance a team's performance metrics.

In [399]:
df_train['GD'] = df_train['FTHG'] - df_train['FTAG']

df_train.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HC,AC,HF,AF,HY,AY,HR,AR,NumDays,GD
0,2000-08-19,Charlton,Man City,4,0,H,2,0,H,Rob Harris,...,6,6,13,12,1,2,0,0,9296,4
1,2000-08-19,Chelsea,West Ham,4,2,H,1,0,H,Graham Barber,...,7,7,19,14,1,2,0,0,9296,2
2,2000-08-19,Coventry,Middlesbrough,1,3,A,1,1,D,Barry Knight,...,8,4,15,21,5,3,1,0,9296,-2
3,2000-08-19,Derby,Southampton,2,2,D,1,2,A,Andy D'Urso,...,5,8,11,13,1,1,0,0,9296,0
4,2000-08-19,Leeds,Everton,2,0,H,2,0,H,Dermot Gallagher,...,6,4,21,20,1,3,0,0,9296,2


### Creating a Season Indicator Feature

The data English Premier League provided spans 10+ years, it is important for this project that we can easliy identify which match belongs to which season. For eg: matches between August 2014 and May 2015 belong in the 14/15 season. 

**Why is this an important feature?**

This will allow us to later apply validation techniques such as *Leave One Season Out*, and without a label identifying which matches belong to which season matches from different seasons can be mixed up, affecting the validation process. 

**Season Indicator logic**

The simplest rule would be to say that if a match is in January – June → it belongs to the previous season and if it is in July – December → it belongs to the season starting that year.

In [400]:
# Create a function to assign season label based on match date - in accordance to feature logic explained above.

def season_name(date):
    month = date.month
    year = date.year
    
    if month >= 8:          
        return year 
    else:                   
        return year - 1
    
# Apply the function to DataFrame
df_train['Season'] = df_train['Date'].apply(season_name)
df_new_features['Season'] = df_new_features['Date'].apply(season_name)

# Confirm
df_train[['Date', 'Season']].head()

,Date,Season
0,2000-08-19,2000
1,2000-08-19,2000
2,2000-08-19,2000
3,2000-08-19,2000
4,2000-08-19,2000


Run a few sanity checks to ensure everything is working properly.

In [401]:
# Check the total number of seasons
df_train['Season'].unique()

# Count the total number of matches per season 
# Note: each season should show 380 matches
df_train['Season'].value_counts().sort_index()

Season
2000    380
2001    380
2002    380
2003    380
2004    380
2005    380
2006    380
2007    380
2008    380
2009    380
2010    380
2011    380
2012    380
2013    380
2014    380
2015    380
2016    380
2017    380
2018    380
2019    380
2020    380
2021    380
2022    380
2023    480
2024    380
Name: count, dtype: int64

## Constructing The Design Matrix For GLM

In this section I will represent the football matches numerically so that they can be used by the Generalised Linear Model (GLM). As this model is fully numerical, we can not give it strings.

**How will the matrix work?**

For every match:
- the home team will be assigned a '+1'.
- the away team will be assigned a '-1'.
- every other team will be assigned a '0'.

### Extract Team List & Create Index Mapping

In order to build the GLM model we must first identify all the unique teams that appear in our data set. We then create a dictionary which maps team names to their unique column indices - so that we can refer to them in the matrix.

Note: This matrix will include all the teams in the dataset, those that got relegated and promoted. Furthermore, I have checked if there is a team which is new to the EPL with no historical data in our dataset - there wasn't. 

In [402]:
# Extracting all team names from dataset
home_team_names = df_train['HomeTeam'].dropna().unique().tolist()

away_team_names = df_train['AwayTeam'].dropna().unique().tolist()

team_names = sorted(list(set(home_team_names+away_team_names))) # used the set function to remove duplicates 

# Create a Dictionary to map unique team names to unique indices
team_index = {}
i = 0

for team in team_names:
    team_index[team] = i
    i += 1

# Check results
print("Number of teams:", len(team_names))
team_index

Number of teams: 46


{'Arsenal': 0,
 'Aston Villa': 1,
 'Birmingham': 2,
 'Blackburn': 3,
 'Blackpool': 4,
 'Bolton': 5,
 'Bournemouth': 6,
 'Bradford': 7,
 'Brentford': 8,
 'Brighton': 9,
 'Burnley': 10,
 'Cardiff': 11,
 'Charlton': 12,
 'Chelsea': 13,
 'Coventry': 14,
 'Crystal Palace': 15,
 'Derby': 16,
 'Everton': 17,
 'Fulham': 18,
 'Huddersfield': 19,
 'Hull': 20,
 'Ipswich': 21,
 'Leeds': 22,
 'Leicester': 23,
 'Liverpool': 24,
 'Luton': 25,
 'Man City': 26,
 'Man United': 27,
 'Middlesbrough': 28,
 'Newcastle': 29,
 'Norwich': 30,
 "Nott'm Forest": 31,
 'Portsmouth': 32,
 'QPR': 33,
 'Reading': 34,
 'Sheffield United': 35,
 'Southampton': 36,
 'Stoke': 37,
 'Sunderland': 38,
 'Swansea': 39,
 'Tottenham': 40,
 'Watford': 41,
 'West Brom': 42,
 'West Ham': 43,
 'Wigan': 44,
 'Wolves': 45}

### Build the GLM Design Matrix & Target Vector
In this section I will convert each football match into numbers, so that I can input them into the model later. We must create the Design Matrix, *X*, where each row represents one match and each column represents a unique team and the Target vector, *y*, which is a list of the goal differences for each match. Note: *X* and *y* should have the same number of rows.

**Design Matrix Logic:**

For every match:
- the home team will be assigned a '+1'.
- the away team will be assigned a '-1'.
- every other team will be assigned a '0'.

The GLM uses *X* and *y* together, and fits the model using a maximum likelihood estimation to calculate a strength value for each team across all matches. 

In [403]:
def glm_data(matches_df, team_names, team_index):
    
    # Removes any empty rows (otherise NaN error)
    matches_df = matches_df.dropna(subset=['HomeTeam', 'AwayTeam'])
    
    # No. of matches & teams
    no_matches = len(matches_df)
    no_teams = len(team_names)
    
    # Create an empty matrix, X
    X = np.zeros((no_matches, no_teams))

    # y is the GD column from the DataFrame
    y = matches_df['GD'].values

    # Ensure index is formatted properly (numbered in order)
    matches_df = matches_df.reset_index(drop=True)

    # Fill X row by row
    for m in range(no_matches):
        # Reads the home and away teams' names
        h_team = matches_df.loc[m, 'HomeTeam']
        a_team = matches_df.loc[m, 'AwayTeam']

        # Find the teams' index number
        h_col = team_index[h_team]
        a_col = team_index[a_team]

        # Sets a value of '+1' for home, and '-1' for away
        X[m, h_col] = 1.0
        X[m, a_col] = -1.0

    return X, y


# Sanity Check
X_all, y_all = glm_data(df_train, team_names, team_index)
print("Design matrix dimensions:", X_all.shape)
print("Target vector dimensions:", y_all.shape)


Design matrix dimensions: (9600, 46)
Target vector dimensions: (9600,)


### Identifiability

This is a unique problem of this feature. Recall that the GLM equation is given by:

$$
\theta_{\text{home}} - \theta_{\text{away}}
$$

The model can only understand the differences between strengths, so any values which keep the differences the same between two teams are equally valid - giving the model infinite strength solutions. 

Example:

Consider teams A & B:
- Strength(A) = 1.5
- Strength(B) = 0.5
- GD = 1.0

Now add '+10' to both A & B's strengths:
- Strength(A) = 11.5
- Strength(B) = 10.5
- GD = 1.0 (stays the same)

These are both equally valid solutions - so how can the model determine which solution is the correct one? The solution to this problem is to choose one team (ideally a mid table team) and set its strength value to equal '0', this becomes the reference team and all other teams' strengths are measured relative to this team's strength. This allows the model to have one unique strength solution per team.

In [404]:
# Choose any team as reference - choosing last team in DataFrame for ease
ref_team = team_names[-1]
print("The Chosen Reference team with strength = 0 is:", ref_team)

# Determine its column index
ref_index = team_index[ref_team]

# Remove that column from our design matrix, X
X_new = np.delete(X_all, ref_index, axis=1)

# Sanity Check
print("Original X dimensions:", X_all.shape)
print("New X dimensions:", X_new.shape)


The Chosen Reference team with strength = 0 is: Wolves
Original X dimensions: (9600, 46)
New X dimensions: (9600, 45)


## Finding a Strength Value For Each Team

This is done by:
- Fitting a linear regression model
- Translating the model's coefficients to team strength values

### Fitting the GLM

So far, I have created a design matrix, *X*, and a goal differences vector, *y*, so we can fit a simple linear regression model to obtain each team's strength.
<br></br>
<center>Predicted GD ≈ $ \theta_{\text{home}} - \theta_{\text{away}} + b$ </center>

where:
- $\theta_{\text{home}}$ is the home team's strength value
- $\theta_{\text{away}}$ is the away team's strength value
- $b$ is the y-intercept of the linear regression model

The model uses maximum likelihood estimation to choose values for all the teams' strengths so that the predicted GD value is as close as possible to the real GD in the data provided.  

We use a Gaussian GLM here because GD values can be estimated with a normal distribution curve over many matches.

In [405]:
# Add an intercept to the reduced design matrix - this is just a column of contants
X_glm = sm.add_constant(X_new)

# Fit a GLM 
model = sm.GLM(y_all, X_glm, family=sm.families.Gaussian())

# Store the results 
results = model.fit()

# Sanity Check - no. of parameters learned by the model
print("Number of parameters learned:", len(results.params))
print(results.params[:10])


Number of parameters learned: 46
[ 0.35143512  1.25278302  0.30668113  0.13324356  0.26173282 -0.14814163
  0.16173282  0.00607259 -0.62494766  0.51109112]


### Assigning Strength Values to Each Team
In the previous step we fit the GLM, the model was able to learn a list of numbers called the *'parameters'*. The first number in that list is the intercept, and the rest of the numbers correspond to the team strength coefficients in order of index, except for the reference team which has been extracted. This team's strength is set as *'0'*. So, a team with a positive strength value is predicted to be stronger than the reference team and a team with a negative value is predicted to be weaker.

So what we need to do is extract the intercept value from the list and assign the remaining parameters (strength coefficients) to their respective teams and store this in a new DataFrame. By the end we should have one strength value per team.

In [406]:
# Defining the intercept parameters from the fitted model
parameters = results.params
intercept = parameters[0]
strength_c = parameters[1:]

# Quick Sanity check
print("Number of team strength coefficients:", len(strength_c))
print("Number of team colums in the reduced design matrix:", X_new.shape[1]) # Both values should be equal

# Create an empty dictionary to place all the teams and strength coefficients in
team_strengths = {}
c_position = 0              # This is to track the position inside the full list of teams

for i in team_names:
    if i == ref_team:
        team_strengths[i] = 0.0
    else: 
        team_strengths[i] = strength_c[c_position]
        c_position += 1

#  Another Sanity Check 3dzzza
print("Number of teams:", len(team_names))
print("Number of strength values:", len(team_strengths))

# Now we convert this dictionary into a DataFrame
teamStrengths_df = pd.DataFrame({
    'Team': list(team_strengths.keys()),
    'Strength Coefficient': list(team_strengths.values())
})

# Sort the teams from highest strength values (strongest) to lowest strength value (weakest)
teamStrengths_df = teamStrengths_df.sort_values('Strength Coefficient', ascending=False).reset_index(drop=True)

# Quick Sanity Check
print(teamStrengths_df)

Number of team strength coefficients: 45
Number of team colums in the reduced design matrix: 45
Number of teams: 46
Number of strength values: 46
                Team  Strength Coefficient
0            Arsenal              1.252783
1          Liverpool              1.250414
2           Man City              1.242620
3            Chelsea              1.218518
4         Man United              1.189340
5          Tottenham              0.759700
6          Brentford              0.511091
7            Everton              0.420689
8          Newcastle              0.352724
9        Aston Villa              0.306681
10          Brighton              0.291545
11         Leicester              0.280069
12         Blackburn              0.261733
13             Leeds              0.230041
14     Middlesbrough              0.204626
15          West Ham              0.196114
16     Nott'm Forest              0.169548
17    Crystal Palace              0.169491
18            Bolton              0.1

## Validation

In the previous section I calculated a single strength value for each team using the entire dataset. Although, this gives a good estimate of the overall team strength, it has a flaw. It uses future matches to describe past matches - this is known as a *'data leakage'*.

For example: if we let the model use results from the 2023/24 season to predict match outcomes of the 2010/11 season, we would be letting the model "see the future" before predicting the present. This is means the model basically cheated by peeking into future results to predict a present match. This would give inflated accuracy numbers as we validate the model, but will fail once we test it on future data (Jan 2026 matches).

Hence, to build a more realistic model, we must ensure that every match only uses match data that was available to it up that point, and nothing beyond it. We do this be generating a feature called *'Time Aware Team Strengths'*. We do this by:

- We sort all the match data by date (and season). (Already Done)
- For a given season, $S_{k}$, we calculate the team strength values using only the seasons before it.
- We then assign this strength, $S_{k}$, to all the matches in that season, $S_{k}$.
- For the first season in the dataset we assign neutral strength values of 0.

This methodology ensures that each match in the dataset is described using only the match data that would have been available up to that point, which should increase predictive accuracy for gameweek 24 (Jan 31st). 

Furthermore, I have decided to weigh team strengths based on recency. This will use the same methodology as before, but it will assign higher weights to more recent seasons, and lower weights to older seasons. I will do this by applying an exponential decay factor of 0.8 per season so that recent seasons exert a greater influence on the GLM fit.

I should then compare models; one which use this recency bias and one which doesn't and evaluate differences in predictive accuracy between the models to determine whether this would improve accuracy or not.

In [407]:
# Obtain a sorted set of all the seasons
# First remove any empty rows
df_train = df_train.dropna(subset=['Season']).reset_index(drop=True)
seasons = sorted(df_train['Season'].unique())

# Create an empty list which will store all team strength values for different seasons
all_strengths = []
decay_factor = 0.8

# Create a for loop to loop through each season one by one

for i, season in enumerate(seasons):
    print("\n\nProcessing season:", season)
    
    prev_seasons = seasons[:i]
    print("Previous season data which is used for this season:", prev_seasons)
    
    # Case 1: No previous data - season 2000
    if not prev_seasons:
        print("There is no past data available for this season, so will assign '0.0' strength value to all teams.")
        for team in team_names:
            all_strengths.append({
                'Season': season,
                'Team': team,
                'Strength Coefficient': 0.0
            })
        continue
    # Case 2: Previous data available - all other seasons
    else:
        prev_df = df_train[df_train['Season'].isin(prev_seasons)].copy()
        print("No. of previous matches used to estimate strength values:", len(prev_df))
    
        # Computing recency-bias by adding higher weights to recent seasons
        # Find the age of the data compared to current season
        prev_df.loc[:, 'Season Age'] = season - prev_df['Season']

        # Calculate the weight of that season
        prev_df.loc[:, 'Weight'] = decay_factor ** prev_df['Season Age']
    
        # Build the design matrix for previous seasons
        X_prev, y_prev = glm_data(prev_df, team_names, team_index)
        print("X_prev dimensions:", X_prev.shape, "\ny_prev length:", len(y_prev))
    
    # Arrange the weights in a vector of same dimensions as y
    weights = prev_df['Weight'].values
    
    # Remove the reference team's column
    ref_index = team_index[ref_team]
    X_prev_red = np.delete(X_prev, ref_index, axis=1)
    print("X_prev_red dimensions:", X_prev_red.shape)
    X_prev_glm = sm.add_constant(X_prev_red)
    
    # Fit the GLM with weights
    model_prev = sm.GLM(
        y_prev,
        X_prev_glm,
        family = sm.families.Gaussian(),
        freq_weights = weights 
    )
    results_prev = model_prev.fit()
    

    # Obtain parameters
    params_prev = results_prev.params
    intercept_prev = params_prev[0]
    teamStrength_coefs_prev = params_prev[1:]
    
    print("Intercept value for previous season's model:", intercept_prev)
    print("No. of team strength coefficients obtained:", len(teamStrength_coefs_prev))
    
    # Redetermine the strengths for all the teams this season
    season_strengths = {}
    pos_coef = 0
    
    for t in team_names:
        if t == ref_team:
            season_strengths[t] = 0.0
        else:
            season_strengths[t] = teamStrength_coefs_prev[pos_coef]
            pos_coef += 1
    
    # Store one row per (Season, Team)
    for t in team_names:
        all_strengths.append({
            'Season': season,
            'Team': t,
            'Strength Coefficient': season_strengths[t]
        })
    



Processing season: 2000
Previous season data which is used for this season: []
There is no past data available for this season, so will assign '0.0' strength value to all teams.


Processing season: 2001
Previous season data which is used for this season: [2000]
No. of previous matches used to estimate strength values: 380
X_prev dimensions: (380, 46) 
y_prev length: 380
X_prev_red dimensions: (380, 45)
Intercept value for previous season's model: 0.4789473684210529
No. of team strength coefficients obtained: 45


Processing season: 2002
Previous season data which is used for this season: [2000, 2001]
No. of previous matches used to estimate strength values: 760
X_prev dimensions: (760, 46) 
y_prev length: 760
X_prev_red dimensions: (760, 45)
Intercept value for previous season's model: 0.37807017543859645
No. of team strength coefficients obtained: 45


Processing season: 2003
Previous season data which is used for this season: [2000, 2001, 2002]
No. of previous matches used to esti

In [408]:
# Add the time-based team-strengths to the main DataFrame

for col in ['HomeTeamStrength', 'AwayTeamStrength', 'StrengthDifference']:
    if col in df_train.columns:
        del df_train[col]

# Convert the all_strengths list into a DataFrame
season_strengths_df = pd.DataFrame(all_strengths)

# Create a home team's strength table - rename the columns
home_strengths_df = season_strengths_df.rename(
    columns={
        'Team': 'HomeTeam',
        'Strength Coefficient': 'HomeTeamStrength'
    }
)

# Add home team strength values into the Data Frame
df_new_features = df_new_features.merge(
    home_strengths_df,
    on=['Season', 'HomeTeam'],
    how='left'
)

# Create an away team's strength table - rename the columns
away_strengths_df = season_strengths_df.rename(
    columns={
        'Team': 'AwayTeam',
        'Strength Coefficient': 'AwayTeamStrength'
    }
)

# Add away team strength values into the Data Frame
df_new_features = df_new_features.merge(
    away_strengths_df,
    on=['Season', 'AwayTeam'],
    how='left'
)

# Calculate the Strength Difference between the home and away teams
df_new_features['StrengthDifference'] = df_new_features['HomeTeamStrength'] - df_new_features['AwayTeamStrength']

# 7. Sanity checks
print("\nMerged training data with time-aware strengths:")
display(df_new_features)

print("\nMissing values check:")
print(df_new_features[['HomeTeamStrength', 'AwayTeamStrength']].isna().sum())




Merged training data with time-aware strengths:


,Date,NumDays,HomeTeam,AwayTeam,FTR,H2H,RecentWH,RecentWA,RecentGDH,RecentGDA,...,ShotAllowedDiff,AttackBalance,DefenseBalance,home_elo_pre,away_elo_pre,elo_diff_pre,Season,HomeTeamStrength,AwayTeamStrength,StrengthDifference
0,2000-08-19,9296,Charlton,Man City,H,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.0,0.0,1500.000000,1500.000000,0.000000,2000,0.000000,0.000000,0.000000
1,2000-08-19,9296,Chelsea,West Ham,H,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.0,0.0,1500.000000,1500.000000,0.000000,2000,0.000000,0.000000,0.000000
2,2000-08-19,9296,Coventry,Middlesbrough,A,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.0,0.0,1500.000000,1500.000000,0.000000,2000,0.000000,0.000000,0.000000
3,2000-08-19,9296,Derby,Southampton,D,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.0,0.0,1500.000000,1500.000000,0.000000,2000,0.000000,0.000000,0.000000
4,2000-08-19,9296,Leeds,Everton,H,0.500000,0.500000,0.500000,0.000000,0.000000,...,0.0,0.0,0.0,1500.000000,1500.000000,0.000000,2000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9595,2025-05-25,251,Ipswich,West Ham,A,0.000282,0.588889,0.388889,0.266667,-0.222222,...,-0.7,0.1,2.5,1351.780976,1772.479113,-420.698137,2024,0.062022,0.154372,-0.092350
9596,2025-05-25,251,Fulham,Man City,A,0.004437,0.502174,0.544444,-0.152174,0.044444,...,-2.7,-0.2,2.6,1677.011651,1577.256487,99.755164,2024,-0.029682,1.763041,-1.792723
9597,2025-05-25,251,Bournemouth,Leicester,H,0.562270,0.355556,0.644444,-0.955556,0.066667,...,-1.4,-0.9,2.8,1607.715011,1673.309574,-65.594563,2024,-0.263437,0.373538,-0.636975
9598,2025-05-25,251,Liverpool,Crystal Palace,D,0.729806,0.722222,0.355556,0.466667,-0.355556,...,-4.8,0.0,2.8,1549.451178,1695.466458,-146.015280,2024,1.347241,0.086672,1.260569



Missing values check:
HomeTeamStrength    0
AwayTeamStrength    0
dtype: int64


Now we rename some of the variables to remain consistent with the brief's naming convention.

In [409]:
df_new_features["HomeStrength"] = df_new_features["HomeTeamStrength"]
df_new_features["AwayStrength"] = df_new_features["AwayTeamStrength"]
df_new_features["StrengthDiff"] = df_new_features["HomeStrength"] - df_new_features["AwayStrength"]

# Sanity Check
df_new_features[["HomeTeam", "AwayTeam", "HomeStrength", "AwayStrength", "StrengthDiff"]]

,HomeTeam,AwayTeam,HomeStrength,AwayStrength,StrengthDiff
0,Charlton,Man City,0.000000,0.000000,0.000000
1,Chelsea,West Ham,0.000000,0.000000,0.000000
2,Coventry,Middlesbrough,0.000000,0.000000,0.000000
3,Derby,Southampton,0.000000,0.000000,0.000000
4,Leeds,Everton,0.000000,0.000000,0.000000
...,...,...,...,...,...
9595,Ipswich,West Ham,0.062022,0.154372,-0.092350
9596,Fulham,Man City,-0.029682,1.763041,-1.792723
9597,Bournemouth,Leicester,-0.263437,0.373538,-0.636975
9598,Liverpool,Crystal Palace,1.347241,0.086672,1.260569


Remove duplicate rows

In [410]:
df_new_features = df_new_features.drop_duplicates(subset=['Date', 'HomeTeam', 'AwayTeam'])

Must check that there are no empty cells (NaN) before passing the data into the model.

In [411]:
print("NaN count per column:")
print(df_new_features.isna().sum())

print("Duplicate rows:", df_new_features.duplicated(subset=['Date','HomeTeam','AwayTeam']).sum())

print(df_new_features.dtypes)

NaN count per column:
Date                  0
NumDays               0
HomeTeam              0
AwayTeam              0
FTR                   0
H2H                   0
RecentWH              0
RecentWA              0
RecentGDH             0
RecentGDA             0
HomeWinRate           0
AwayWinRate           0
AvgGF_H               0
AvgGA_H               0
AvgSF_H               0
AvgSA_H               0
AvgGF_A               0
AvgGA_A               0
AvgSF_A               0
AvgSA_A               0
FormDiff              0
GDDiff                0
ShotDiff              0
ShotAllowedDiff       0
AttackBalance         0
DefenseBalance        0
home_elo_pre          0
away_elo_pre          0
elo_diff_pre          0
Season                0
HomeTeamStrength      0
AwayTeamStrength      0
StrengthDifference    0
HomeStrength          0
AwayStrength          0
StrengthDiff          0
dtype: int64
Duplicate rows: 0
Date                  datetime64[ns]
NumDays                        int64
HomeTeam 

In [412]:
df_new_features

,Date,NumDays,HomeTeam,AwayTeam,FTR,H2H,RecentWH,RecentWA,RecentGDH,RecentGDA,...,home_elo_pre,away_elo_pre,elo_diff_pre,Season,HomeTeamStrength,AwayTeamStrength,StrengthDifference,HomeStrength,AwayStrength,StrengthDiff
0,2000-08-19,9296,Charlton,Man City,H,0.500000,0.500000,0.500000,0.000000,0.000000,...,1500.000000,1500.000000,0.000000,2000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2000-08-19,9296,Chelsea,West Ham,H,0.500000,0.500000,0.500000,0.000000,0.000000,...,1500.000000,1500.000000,0.000000,2000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2000-08-19,9296,Coventry,Middlesbrough,A,0.500000,0.500000,0.500000,0.000000,0.000000,...,1500.000000,1500.000000,0.000000,2000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,2000-08-19,9296,Derby,Southampton,D,0.500000,0.500000,0.500000,0.000000,0.000000,...,1500.000000,1500.000000,0.000000,2000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2000-08-19,9296,Leeds,Everton,H,0.500000,0.500000,0.500000,0.000000,0.000000,...,1500.000000,1500.000000,0.000000,2000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9595,2025-05-25,251,Ipswich,West Ham,A,0.000282,0.588889,0.388889,0.266667,-0.222222,...,1351.780976,1772.479113,-420.698137,2024,0.062022,0.154372,-0.092350,0.062022,0.154372,-0.092350
9596,2025-05-25,251,Fulham,Man City,A,0.004437,0.502174,0.544444,-0.152174,0.044444,...,1677.011651,1577.256487,99.755164,2024,-0.029682,1.763041,-1.792723,-0.029682,1.763041,-1.792723
9597,2025-05-25,251,Bournemouth,Leicester,H,0.562270,0.355556,0.644444,-0.955556,0.066667,...,1607.715011,1673.309574,-65.594563,2024,-0.263437,0.373538,-0.636975,-0.263437,0.373538,-0.636975
9598,2025-05-25,251,Liverpool,Crystal Palace,D,0.729806,0.722222,0.355556,0.466667,-0.355556,...,1549.451178,1695.466458,-146.015280,2024,1.347241,0.086672,1.260569,1.347241,0.086672,1.260569


# **Model training**

In [413]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder


Use label encoding to convert team names into numeric values and maps the match result ('FTR') to integers for model training.

In [414]:
#Map teams to a number list
le1 = LabelEncoder()
all_teams = pd.concat([df_new_features['HomeTeam'], df_new_features['AwayTeam']]).astype(str)
le1.fit(all_teams)
df_new_features['HomeTeam'] = le1.transform(df_new_features['HomeTeam'].astype(str))
df_new_features['AwayTeam'] = le1.transform(df_new_features['AwayTeam'].astype(str))

#Map final result to a number list
mapping = {"H":0,"D":1,"A":2 }
df_new_features['FTR'] = df_new_features['FTR'].map(mapping)

In [415]:
print("NaN count per column:")
print(df_new_features.isna().sum())

print("Duplicate rows:", df_new_features.duplicated(subset=['Date','HomeTeam','AwayTeam']).sum())

print(df_new_features.dtypes)

NaN count per column:
Date                  0
NumDays               0
HomeTeam              0
AwayTeam              0
FTR                   0
H2H                   0
RecentWH              0
RecentWA              0
RecentGDH             0
RecentGDA             0
HomeWinRate           0
AwayWinRate           0
AvgGF_H               0
AvgGA_H               0
AvgSF_H               0
AvgSA_H               0
AvgGF_A               0
AvgGA_A               0
AvgSF_A               0
AvgSA_A               0
FormDiff              0
GDDiff                0
ShotDiff              0
ShotAllowedDiff       0
AttackBalance         0
DefenseBalance        0
home_elo_pre          0
away_elo_pre          0
elo_diff_pre          0
Season                0
HomeTeamStrength      0
AwayTeamStrength      0
StrengthDifference    0
HomeStrength          0
AwayStrength          0
StrengthDiff          0
dtype: int64
Duplicate rows: 0
Date                  datetime64[ns]
NumDays                        int64
HomeTeam 

Select all columns except 'Date' and 'FTR' as features for model training, and sets 'FTR' as the target variable.

In [416]:
feature_cols = [
    col for col in df_new_features.columns
    if col not in ['Date', 'FTR']
]
X = df_new_features[feature_cols]
y = df_new_features['FTR']

Split the dataset into training and validation sets, keeping the class distribution balanced by using stratified sampling.

In [417]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Train an XGBoost multi-class classifier to predict football match outcomes using engineered features.

In [418]:
model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class= 3,
    random_state=42,
    use_label_encoder=False,
    eval_metric="mlogloss"
)
model.fit(X_train, y_train)

/opt/anaconda3/envs/ucl_cs_ml_module/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [18:42:07] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None, num_class=3, ...)

In [419]:
y_pred = model.predict(X_val)
print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred, target_names=["Home Win", "Draw", "Away Win"]))

Accuracy: 0.5184210526315789
              precision    recall  f1-score   support

    Home Win       0.55      0.80      0.65       870
        Draw       0.27      0.07      0.11       469
    Away Win       0.50      0.46      0.48       561

    accuracy                           0.52      1900
   macro avg       0.44      0.44      0.41      1900
weighted avg       0.46      0.52      0.47      1900



Brier score calculation

In [420]:
from sklearn.metrics import brier_score_loss
import numpy as np

y_proba = model.predict_proba(X_val)

brier_scores = []
for i in range(y_proba.shape[1]):
    brier = brier_score_loss((y_val == i).astype(int), y_proba[:, i])
    brier_scores.append(brier)
print("Brier scores for each class:", brier_scores)
print("Mean Brier score:", np.mean(brier_scores))

Brier scores for each class: [0.22239092114322148, 0.1873497579682363, 0.18652358792704948]
Mean Brier score: 0.1987547556795024
